In [16]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Cargar datos
spark = SparkSession.builder \
    .appName("Analisis_Categorias") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_scraping") \
    .config("spark.mongodb.write.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_libros") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.6.1") \
    .getOrCreate()

df = spark.read.format("mongodb").load()

In [17]:
df.printSchema()
df.show(5)

root
 |-- _id: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- item: string (nullable = true)
 |-- valor: double (nullable = true)

+--------------------+---------+-------------------+--------+--------------------+-----+
|                 _id|categoria|      fecha_captura|   grupo|                item|valor|
+--------------------+---------+-------------------+--------+--------------------+-----+
|69de599d111752179...|   Travel|2026-04-14 15:13:33|G1_Libro|  It's Only Himalaya|45.17|
|69de599d111752179...|   Travel|2026-04-14 15:13:33|G1_Libro|Full Moon over No...|49.43|
+--------------------+---------+-------------------+--------+--------------------+-----+



In [18]:
from pyspark.sql.functions import col

# Solo queremos registros que tengan un item real y un valor mayor a 0
df_filtrado = df.filter((col("item").isNotNull()) & (col("valor") > 0))

print("PASO 2: Limpieza completada.")
print("Registros originales:", df.count())
print("Registros validos:", df_filtrado.count())

PASO 2: Limpieza completada.
Registros originales: 2
Registros validos: 2


In [19]:
df.select("item", "valor").show()

+--------------------+-----+
|                item|valor|
+--------------------+-----+
|  It's Only Himalaya|45.17|
|Full Moon over No...|49.43|
+--------------------+-----+



In [20]:
df.filter(df["valor"] > 50).show()

+---+---------+-------------+-----+----+-----+
|_id|categoria|fecha_captura|grupo|item|valor|
+---+---------+-------------+-----+----+-----+
+---+---------+-------------+-----+----+-----+



In [21]:
df.groupBy("grupo").count().show()

+--------+-----+
|   grupo|count|
+--------+-----+
|G1_Libro|    2|
+--------+-----+



In [22]:
# 2. Transformacion y Agregacion por CATEGORIA
# Esto permite comparar que generos son mas caros en promedio
reporte_categorias = df.groupBy("categoria").agg(
    F.count("item").alias("cantidad_libros"),
    F.avg("valor").alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("ANALISIS DE MERCADO POR CATEGORIA:")
reporte_categorias.show()

ANALISIS DE MERCADO POR CATEGORIA:
+---------+---------------+---------------+-------------+-------------+
|categoria|cantidad_libros|precio_promedio|precio_minimo|precio_maximo|
+---------+---------------+---------------+-------------+-------------+
|   Travel|              2|           47.3|        45.17|        49.43|
+---------+---------------+---------------+-------------+-------------+



In [23]:
fila = df.orderBy(F.desc("valor")).select(
    "item", "categoria", "fecha_captura", "valor"
).first()

print("Producto más caro:", fila["item"])
print("Categoría:", fila["categoria"])
print("Fecha y hora de captura:", fila["fecha_captura"])
print("Valor:", fila["valor"])

Producto más caro: Full Moon over Noah’s Ark
Categoría: Travel
Fecha y hora de captura: 2026-04-14 15:13:33
Valor: 49.43
